### Using the HadISD datset (version 3.4.0.2023f) with PyEarthTools
HadISD is a global sub-daily dataset based on the ISD dataset from NOAA's NCEI. As well as station selection criteria, a suite of quality control tests has been run on the major climatological variables.

The dataset can be downloaded here: https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

In [1]:
import datetime

import pyearthtools.pipeline as petpipe
import pyearthtools.data as petdata
import pyearthtools.tutorial


In [2]:
# train/validation/test split dates
train_start = "1970-01-01T00"
train_end = "2022-12-31T23"

In [ ]:
varname_val_map = {
        "total_cloud_cover": -999., 
        "low_cloud_cover": -999., 
        "mid_cloud_cover": -999.,
        "high_cloud_cover": -999.
    }

# Probably sensible to accetp a list since some variables have multiple values
# This would look like:
# varname_val_map = {
#         "total_cloud_cover": [-888., -999.],
#         "low_cloud_cover": [-888., -999.],
#         "mid_cloud_cover": [-999.],
#         "high_cloud_cover": [-999.]
#     }
# This is a list of variables to be used in the pipeline
# and the corresponding values to be masked
# in the data

In [ ]:


flagged_labels = [
        'temperatures', 'dewpoints', 'slp',
        'stnlp', 'windspeeds', 'winddirs', 
        'total_cloud_cover', 'low_cloud_cover', 'mid_cloud_cover', 
        'high_cloud_cover', 'precip1_depth', 'precip2_depth', 
        'precip3_depth', 'precip6_depth', 'precip9_depth',
        'precip12_depth', 'precip15_depth', 'precip18_depth', 
        'precip24_depth'
    ]

date_range=(datetime.datetime(1986,11,20,12,0), datetime.datetime(1986,11,21,0,0))

In [ ]:
data_prep_pipe = petpipe.Pipeline(
    # petdata.archive.hadisd(("010010-99999"), variables = ["total_cloud_cover", "temperatures", "flagged_obs"]),
    # petdata.transforms.values.SetMissingToNaN(varname_val_map),
    petdata.archive.hadisd(["010014-99999", "010010-99999", "010030-99999"]),    
    petdata.transforms.values.AddFlaggedObs(flagged_labels),
    petdata.transforms.values.SetMissingToNaN(varname_val_map),
    # petdata.transforms.coordinates.ReIndexTime(date_range),

    # petdata.archive.hadisd("010014-99999", variables = ["slp", "dewpoints", "temperatures", "station_id"]),
    # petdata.archive.hadisd(country_code = "24", station_id = "010010", nearest_neighbors = 12, lon= 0.0, lat = 0.0), # Geospatial fence 
    # petdata.archive.hadisd("010014-99999"), # Geospatial fence
)
data_prep_pipe

# Don't know what combination of stations you want when selecting nearest neighbours, so want to cahce on a per-station basis


In [ ]:
ds = data_prep_pipe["1986-11-20T12"]
ds

In some cases you will get the following error when passing a date time to a pipeline object: `IndexWarning: Could not find time in dataset to select on. Petdt('1931-01-01T07')`<br>

This indicates that data for the datetime you chose does not exist. In this case PET will load all data from your station selection.

In [ ]:
# show total_cloud_cover from ds
tcc = ds["total_cloud_cover"]
tcc

# Use the section below to build tests to check that all pipeline steps are working as expected

## Test SetMissingToNaN

In [ ]:
import xarray as xr
from pyearthtools.data.transforms.values import SetMissingToNaN
import numpy as np

def test_set_missing_to_nan():
    data = xr.Dataset({
        "total_cloud_cover": ("time", [0, -999, 50]),
        "low_cloud_cover": ("time", [10, -999, 20]),
    })

    varname_val_map = {
        "total_cloud_cover": -99.0,
        "low_cloud_cover": -999.0,
    }

    transform = SetMissingToNaN(varname_val_map)
    transformed_data = transform.apply(data)

    if np.isnan(transformed_data["total_cloud_cover"].data[1]):
        print("Total cloud cover at index 1 is NaN")
    else:
        print("Total cloud cover at index 1 is not NaN")
    
    if np.isnan(transformed_data["low_cloud_cover"].data[1]):
        print("Low cloud cover at index 1 is NaN")
    
    if transformed_data["total_cloud_cover"].data[2] == 50:
        print("Total cloud cover at index 2 is 50")

In [ ]:
test_set_missing_to_nan()

## Test AddFlaggedObs

In [ ]:
import xarray as xr
import numpy as np
from pyearthtools.data.transforms.values import AddFlaggedObs

def test_add_flagged_obs():
    # Mock dataset with flagged observations
    data = xr.Dataset(
        {
            "temperatures": (("time",), [np.nan, 15.0, np.nan]),
            "dewpoints": (("time",), [np.nan, 10.0, np.nan]),
            "flagged_obs": (
                ("time", "flagged"),
                [
                    [20.0, np.nan],  # Flagged data for time=0
                    [np.nan, np.nan],  # No flagged data for time=1
                    [25.0, 12.0],  # Flagged data for time=2
                ],
            ),
        },
        coords={
            "time": [0, 1, 2],
            "flagged": [0, 1],  # Indices for flagged variables
        },
    )

    # Add flagged_value attribute to variables
    data["temperatures"].attrs["flagged_value"] = -999.0
    data["dewpoints"].attrs["flagged_value"] = -999.0

    # Define flagged labels corresponding to the flagged dimension
    flagged_labels = ["temperatures", "dewpoints"]

    # Apply the AddFlaggedObs transform
    transform = AddFlaggedObs(flagged_labels)
    transformed_data = transform.apply(data)

    # Assert that flagged data has been restored
    assert np.allclose(
        transformed_data["temperatures"].data, [20.0, 15.0, 25.0], equal_nan=True
    )
    assert np.allclose(
        transformed_data["dewpoints"].data, [np.nan, 10.0, 12.0], equal_nan=True
    )

    print("Test passed: Flagged observations were correctly restored.")

# Run the test
test_add_flagged_obs()

In [3]:
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

hadisd = HadISDIndex(["010010-99999"])
paths = hadisd.filesystem(["010010-99999"])
paths


{'010010-99999': PosixPath('/Users/joelmiller/Projects/data/hadisd/WMO_000000-029999/hadisd.3.4.0.2023f_19310101-20240101_010010-99999.nc')}

In [4]:
data = hadisd.load(paths)
data

<xarray.Dataset> Size: 252MB
Dimensions:                (coordinate_length: 1, time: 276194, test: 71,
                            flagged: 19, reporting_v: 19, reporting_t: 1116,
                            reporting_2: 2)
Coordinates:
    longitude              (coordinate_length) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
    latitude               (coordinate_length) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
    elevation              (coordinate_length) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
  * time                   (time) datetime64[ns] 2MB 1931-01-01T06:00:00 ... ...
Dimensions without coordinates: coordinate_length, test, flagged, reporting_v,
                                reporting_t, reporting_2
Data variables: (12/27)
    station_id             |S12 12B ...
    temperatures           (time) float64 2MB dask.array<chunksize=(276194,), meta=np.ndarray>
    dewpoints              (time) float64 2MB dask.array<chunksize=(276194,), meta=np.ndarray>
    slp                    (time) float64 2MB dask.array<chunksize=(276194,), meta=np.ndarray>
    stnlp                  (time) float64 2MB dask.array<chunksize=(276194,), meta=np.ndarray>
    windspeeds             (time) float64 2MB dask.array<chunksize=(276194,), meta=np.ndarray>
    ...                     ...
    wind_gust              (time) float64 2MB dask.array<chunksize=(276194,), meta=np.ndarray>
    past_sigwx1            (time) float64 2MB dask.array<chunksize=(276194,), meta=np.ndarray>
    input_station_id       (time) object 2MB dask.array<chunksize=(276194,), meta=np.ndarray>
    quality_control_flags  (time, test) float64 157MB dask.array<chunksize=(69049, 18), meta=np.ndarray>
    flagged_obs            (time, flagged) float64 42MB dask.array<chunksize=(138097, 10), meta=np.ndarray>
    reporting_stats        (reporting_v, reporting_t, reporting_2) float64 339kB dask.array<chunksize=(19, 1116, 2), meta=np.ndarray>
Attributes: (12/39)
    title:                       HadISD
    institution:                 Met Office Hadley Centre, Exeter, UK
    source:                      HadISD data product
    references:                  Dunn, 2019, Met Office Hadley Centre Technic...
    creator_name:                Robert Dunn
    creator_url:                 www.metoffice.gov.uk
    ...                          ...
    station_information:         Where station is a composite the station id ...
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    featureType:                 timeSeries
    processing_date:             08-Jan-2024
    history:                     Created by mk_netcdf_files.py \nDuplicate Mo...

In [6]:
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

hadisd = HadISDIndex(["010014-99999", "010010-99999", "010030-99999"])
paths = hadisd.filesystem(["010014-99999", "010010-99999", "010030-99999"])
paths

{'010014-99999': PosixPath('/Users/joelmiller/Projects/data/hadisd/WMO_000000-029999/hadisd.3.4.0.2023f_19310101-20240101_010014-99999.nc'),
 '010010-99999': PosixPath('/Users/joelmiller/Projects/data/hadisd/WMO_000000-029999/hadisd.3.4.0.2023f_19310101-20240101_010010-99999.nc'),
 '010030-99999': PosixPath('/Users/joelmiller/Projects/data/hadisd/WMO_000000-029999/hadisd.3.4.0.2023f_19310101-20240101_010030-99999.nc')}

In [7]:
data = hadisd.load(paths, combine='nested')
data

ValueError: Cannot merge data from files: [PosixPath('/Users/joelmiller/Projects/data/hadisd/WMO_000000-029999/hadisd.3.4.0.2023f_19310101-20240101_010014-99999.nc'), PosixPath('/Users/joelmiller/Projects/data/hadisd/WMO_000000-029999/hadisd.3.4.0.2023f_19310101-20240101_010010-99999.nc'), PosixPath('/Users/joelmiller/Projects/data/hadisd/WMO_000000-029999/hadisd.3.4.0.2023f_19310101-20240101_010030-99999.nc')].
Set `soft_fail` to True to return a dictionary of the data from each of the sources, loaded separately.